# SAM 2 Fine-Tuning Pipeline for Hieroglyphs (Enhanced)

This notebook demonstrates the complete end-to-end pipeline to fine-tune **SAM 2 (Hiera Large)**. 
**Enhancements included:**
- **Validation Split**: Using separate folders for `train`, `valid`, and `test`.
- **Metrics**: Tracking Mean IoU and Pixel Accuracy for both Train and Val sets.
- **Early Stopping**: Automated halt at Epoch 10 (Patience 3) based on Val Loss.
- **Final Reporting**: Dedicated cell to evaluate the best model on the test set.

In [1]:
!pip install git+https://github.com/huggingface/transformers.git accelerate pycocotools -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 102.9 MB/s eta 0:00:0000:01


In [1]:
import os
# Must be set BEFORE torch is imported, otherwise it has no effect
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
import cv2
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import Sam2Model, Sam2Processor
from pycocotools import mask as mask_util
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 1. Metrics and Preprocessing
We implement **mIoU** (Intersection over Union) as our primary accuracy metric and a multi-stage denoising pipeline.

In [3]:
def compute_iou(preds, targets):
    if preds.dim() == 3:   preds   = preds.unsqueeze(1)
    if targets.dim() == 3: targets = targets.unsqueeze(1)
    if preds.shape[-2:] != targets.shape[-2:]:
        targets = torch.nn.functional.interpolate(
            targets.float(), size=preds.shape[-2:], mode='nearest'
        )
    preds   = (torch.sigmoid(preds) > 0.5).float()
    targets = targets.float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union        = (preds + targets).clamp(0, 1).sum(dim=(1, 2, 3))
    return ((intersection + 1e-6) / (union + 1e-6)).mean().item()


def compute_pixel_accuracy(preds, targets):
    if preds.dim() == 3:   preds   = preds.unsqueeze(1)
    if targets.dim() == 3: targets = targets.unsqueeze(1)
    if preds.shape[-2:] != targets.shape[-2:]:
        targets = torch.nn.functional.interpolate(
            targets.float(), size=preds.shape[-2:], mode='nearest'
        )
    preds = (torch.sigmoid(preds) > 0.5).float()
    return (preds == targets.float()).float().mean().item()


def preprocess_image(image_path):
    img = cv2.imread(image_path)
    if img is None: return None
    img = cv2.medianBlur(img, 3)
    img = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    cl = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(l)
    return Image.fromarray(cv2.cvtColor(cv2.merge((cl,a,b)), cv2.COLOR_LAB2RGB))

## 2. Dataset and Dataloaders
Loading data from separate `train`, `valid`, and `test` directories.

In [4]:
class HieroglyphsDataset(Dataset):
    def __init__(self, data_dir, processor):
        self.data_dir   = data_dir
        self.processor  = processor
        self.json_paths = sorted(glob(os.path.join(data_dir, "*.json")))

    def __len__(self):
        return len(self.json_paths)

    def __getitem__(self, idx):
        json_path = self.json_paths[idx]
        img_path  = json_path.replace(".json", ".jpg")
        image     = preprocess_image(img_path)
        if image is None: return None

        with open(json_path, 'r') as f:
            data = json.load(f)

        bboxes, gt_masks = [], []
        for ann in data['annotations']:
            if 'segmentation' not in ann: continue
            x, y, w, h = ann['bbox']
            bboxes.append([x, y, x + w, y + h])
            segm = ann['segmentation']
            if isinstance(segm['counts'], str):
                segm['counts'] = segm['counts'].encode('utf-8')
            gt_masks.append(mask_util.decode(segm))

        if not bboxes: return None
        inputs = self.processor(image, input_boxes=[bboxes], return_tensors="pt")
        inputs["gt_masks"] = torch.tensor(np.array(gt_masks), dtype=torch.float32)
        return {k: v.squeeze(0) if k != "gt_masks" else v for k, v in inputs.items()}


def custom_collate(batch):
    return [b for b in batch if b is not None]


processor    = Sam2Processor.from_pretrained("facebook/sam2.1-hiera-small")
BASE_PATH    = "/kaggle/input/datasets/karimtawfik/hieroglyphics-segmentation-data-sam-2-format/segmentation.v1-2023-03-12-7-33pm.sam2/"
train_loader = DataLoader(HieroglyphsDataset(BASE_PATH+"train", processor), batch_size=2, shuffle=True,  collate_fn=custom_collate)
valid_loader = DataLoader(HieroglyphsDataset(BASE_PATH+"valid", processor), batch_size=2, shuffle=False, collate_fn=custom_collate)
test_loader  = DataLoader(HieroglyphsDataset(BASE_PATH+"test",  processor), batch_size=1, shuffle=False, collate_fn=custom_collate)

processor_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/683 [00:00<?, ?B/s]

## 3. Model and Training Setup
We freeze the vision and prompt encoders to train only the mask decoder.

In [5]:
model     = Sam2Model.from_pretrained("facebook/sam2.1-hiera-small").to(device)
processor = Sam2Processor.from_pretrained("facebook/sam2.1-hiera-small")

# Freeze vision encoder and prompt encoder — only train mask decoder
for name, param in model.named_parameters():
    if name.startswith("vision_encoder") or name.startswith("prompt_encoder"):
        param.requires_grad = False

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
bce_loss  = nn.BCEWithLogitsLoss()

# AMP scaler — fp16 halves activation memory with almost no accuracy loss
scaler = torch.cuda.amp.GradScaler()

# SAM2 decoder does repeat_interleave(n_boxes) on the image embedding.
# Processing boxes in chunks of CHUNK_SIZE lets us train on ALL boxes
# while keeping peak memory proportional to CHUNK_SIZE, not total boxes.
# Reduce to 8 if you still hit OOM.
CHUNK_SIZE = 16


def calc_loss(pred, target):
    target = target.unsqueeze(1) if target.dim() == 3 else target
    target_resized = torch.nn.functional.interpolate(
        target.float(), size=(pred.shape[-2], pred.shape[-1]), mode='nearest'
    )
    return bce_loss(pred, target_resized)

config.json: 0.00B [00:00, ?B/s]

You are using a model of type sam2_video to instantiate a model of type sam2. This is not supported for all configurations of models and can yield errors.


model.safetensors:   0%|          | 0.00/184M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/357 [00:00<?, ?it/s]

/tmp/ipykernel_55/1954711036.py:13: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


## 4. Training with Early Stopping
Monitors `val_loss` with a patience of 3 epochs.

In [ ]:
best_val_loss = float('inf')
patience, patience_counter = 3, 0

for epoch in range(10):
    model.train()
    t_losses, t_ious, t_accs = [], [], []

    for batch in train_loader:
        if not batch: continue

        for item in batch:
            p_val     = item["pixel_values"].unsqueeze(0).to(device)
            boxes_all = item["input_boxes"]   # [N, 4]
            gt_all    = item["gt_masks"]       # [N, H, W]
            n_boxes   = len(boxes_all)
            n_chunks  = max(1, (n_boxes + CHUNK_SIZE - 1) // CHUNK_SIZE)

            optimizer.zero_grad()
            item_loss = 0.0

            for start in range(0, n_boxes, CHUNK_SIZE):
                end          = min(start + CHUNK_SIZE, n_boxes)
                boxes_chunk  = boxes_all[start:end].unsqueeze(0).to(device)
                gt_chunk     = gt_all[start:end].to(device)

                with torch.cuda.amp.autocast():
                    pred = model(
                        pixel_values=p_val,
                        input_boxes=boxes_chunk,
                        multimask_output=False
                    ).pred_masks.squeeze(0)
                    # Divide by n_chunks so overall gradient scale stays consistent
                    loss = calc_loss(pred, gt_chunk) / n_chunks

                scaler.scale(loss).backward()   # gradients accumulate across chunks

                with torch.no_grad():
                    t_ious.append(compute_iou(pred.detach(), gt_chunk))
                    t_accs.append(compute_pixel_accuracy(pred.detach(), gt_chunk))

                item_loss += loss.item()
                del boxes_chunk, gt_chunk, pred, loss
                torch.cuda.empty_cache()

            # Single optimiser step after all chunks for this image
            scaler.step(optimizer)
            scaler.update()
            t_losses.append(item_loss)

            del p_val
            torch.cuda.empty_cache()

    # ── Validation ──────────────────────────────────────────────────────────
    model.eval()
    v_losses, v_ious, v_accs = [], [], []
    with torch.no_grad():
        for batch in valid_loader:
            if not batch: continue
            for item in batch:
                p_val     = item["pixel_values"].unsqueeze(0).to(device)
                boxes_all = item["input_boxes"]
                gt_all    = item["gt_masks"]
                n_boxes   = len(boxes_all)

                for start in range(0, n_boxes, CHUNK_SIZE):
                    end         = min(start + CHUNK_SIZE, n_boxes)
                    boxes_chunk = boxes_all[start:end].unsqueeze(0).to(device)
                    gt_chunk    = gt_all[start:end].to(device)

                    with torch.cuda.amp.autocast():
                        pred = model(
                            pixel_values=p_val,
                            input_boxes=boxes_chunk,
                            multimask_output=False
                        ).pred_masks.squeeze(0)

                    v_losses.append(calc_loss(pred, gt_chunk).item())
                    v_ious.append(compute_iou(pred, gt_chunk))
                    v_accs.append(compute_pixel_accuracy(pred, gt_chunk))

                    del boxes_chunk, gt_chunk, pred
                    torch.cuda.empty_cache()

                del p_val
                torch.cuda.empty_cache()

    # ── Logging ─────────────────────────────────────────────────────────────
    avg_tl, avg_vl = np.mean(t_losses), np.mean(v_losses)
    print(f"Epoch {epoch+1} | T-Loss: {avg_tl:.4f} | T-IoU: {np.mean(t_ious):.4f} | T-Acc: {np.mean(t_accs):.4f} | V-Loss: {avg_vl:.4f} | V-IoU: {np.mean(v_ious):.4f} | V-Acc: {np.mean(v_accs):.4f}")

    if avg_vl < best_val_loss:
        best_val_loss, patience_counter = avg_vl, 0
        model.save_pretrained("/kaggle/working/best_sam_model")
        processor.save_pretrained("/kaggle/working/best_sam_model")
        print("--> Best Model Saved")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early Stopping Triggered.")
            break

/tmp/ipykernel_55/1839413480.py:26: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_55/1839413480.py:70: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch 1 | T-Loss: 3.6197 | T-IoU: 0.5945 | T-Acc: 0.9974 | V-Loss: 0.6230 | V-IoU: 0.4852 | V-Acc: 0.9770


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

--> Best Model Saved


## 5. Final Test Evaluation
Runs the final model on the unseen test set.

In [ ]:
print("Evaluating on TEST set...")
best_model = Sam2Model.from_pretrained("/kaggle/working/best_sam_model").to(device)
test_ious, test_accs = [], []

with torch.no_grad():
    for batch in test_loader:
        if not batch: continue
        item      = batch[0]
        p_val     = item["pixel_values"].unsqueeze(0).to(device)
        boxes_all = item["input_boxes"]
        gt_all    = item["gt_masks"]
        n_boxes   = len(boxes_all)

        for start in range(0, n_boxes, CHUNK_SIZE):
            end         = min(start + CHUNK_SIZE, n_boxes)
            boxes_chunk = boxes_all[start:end].unsqueeze(0).to(device)
            gt_chunk    = gt_all[start:end].to(device)

            with torch.cuda.amp.autocast():
                pred = best_model(
                    pixel_values=p_val,
                    input_boxes=boxes_chunk,
                    multimask_output=False
                ).pred_masks.squeeze(0)

            test_ious.append(compute_iou(pred, gt_chunk))
            test_accs.append(compute_pixel_accuracy(pred, gt_chunk))

            del boxes_chunk, gt_chunk, pred
            torch.cuda.empty_cache()

        del p_val
        torch.cuda.empty_cache()

print(f"FINAL RESULTS | Test mIoU: {np.mean(test_ious):.4f} | Test Pixel Accuracy: {np.mean(test_accs):.4f}")